# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 5:** relationships — old friends · partners · relatives · classmates & dorm-mates · boss & colleagues · landlord & tenant · strangers on train/bus/plane/ferry · people on holiday · a strike · a football match · a children's birthday party

Grammar in context: reciprocal verbs (*ci vediamo, ci siamo lasciati, ci vogliamo bene*), *non… da* + time, *sono contento che* + subjunctive, *mi manchi*, *potresti / potrei*, *Le dispiace se…?*, *ci diamo del tu?* Part 11 reviews sentences from Lessons 1–4.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson05_Relazioni_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 5 (B1)
# Theme: relationships & social situations — friends, partners, relatives,
#        classmates, boss & colleagues, landlord & tenant, dorm-mates,
#        travellers on train/bus/plane/ferry, people on holiday,
#        a strike, a football match, a children's birthday party
# Grammar: reciprocal verbs (ci vediamo, ci siamo lasciati, ci vogliamo bene),
#          non ... da + time, sono contento che + congiuntivo, dovresti / al tuo posto,
#          Le dispiace se...? / Ti dà fastidio se...?, informal imperative, stare con
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson five.
(narrator) Today's lesson is about people, and how we talk to them. Friends, partners, family, colleagues, landlords, and strangers on a train.
(narrator) As always, answer out loud in the pause. Then repeat after the Italian speaker.
(narrator) Listen to this conversation. Luca is on a train. A woman asks if a seat is free. It is Chiara, an old classmate from high school.
(pause 1)
(it M) Chiara? Sei proprio tu? Non ci vediamo da dieci anni!
(it F) Luca! Che sorpresa! Questo posto è libero?
(it M) Certo, siediti. Allora, come stai? Che cosa fai adesso?
(it F) Lavoro in una banca a Milano. Il mio capo è molto esigente, ma i colleghi sono simpatici.
(it M) E sei ancora fidanzata con Paolo?
(it F) No, ci siamo lasciati due anni fa. Adesso sto con Andrea, e ci sposiamo a settembre!
(it M) Ma dai! Congratulazioni! Sono proprio contento che tu sia felice.
(it F) Grazie! E tu? Sei sposato?
(it M) Sì, e abbiamo due figli. Oggi vado alla festa di compleanno di un amichetto di mio figlio.
(it F) Che bello! E vai ancora d'accordo con tua sorella?
(it M) Sì, ci vogliamo molto bene. Con mia suocera, invece, è un'altra storia!
(it F) Dai, dobbiamo restare in contatto. Mi dai il tuo numero?
(it M) Certo. Ti scrivo su WhatsApp stasera.
(pause 1.5)

(section) Part 1 — Old friends
(narrator) Let's look at this conversation, piece by piece.
(narrator) Luca says: we haven't seen each other for ten years. {Vedersi} is a reciprocal verb: to see each other.
(repeat x2 slow) Non ci vediamo da dieci anni!
(prompt) Say: we haven't seen each other for ten years! || Non ci vediamo da dieci anni!
(prompt) Say: we haven't seen each other for a long time. || Non ci vediamo da tanto tempo.
(prompt) Say: what a surprise! || Che sorpresa!
(prompt) Ask: is this seat free? || Questo posto è libero?
(prompt) Answer informally: of course, sit down. || Certo, siediti.
(prompt) Ask a friend: what are you doing now? Talk about work. || Che cosa fai adesso?
(prompt) Say: we were classmates in high school. Use {compagni di classe}. || Eravamo compagni di classe al liceo.
(narrator) To get along with someone is {andare d'accordo con}.
(build) cordo | d'accordo | andare d'accordo
(prompt) Ask: do you still get along with your sister? || Vai ancora d'accordo con tua sorella?
(prompt) Say: we haven't seen each other for ten years! || Non ci vediamo da dieci anni!

(narrator) To love each other, as family or friends, is {volersi bene}. {Ti amo} is only for romantic love.
(repeat x2 slow) Ci vogliamo molto bene.
(prompt) Say: we love each other very much. Talk about you and your sister. || Ci vogliamo molto bene.
(prompt) Say to a friend: I care about you a lot. Use {volere bene}. || Ti voglio tanto bene.
(narrator) A mother-in-law is {la suocera}. A brother-in-law is {il cognato}.
(prompt) Say: with my mother-in-law, however, it's another story! || Con mia suocera, invece, è un'altra storia!
(prompt) Say: my brother-in-law is very nice. || Mio cognato è molto simpatico.
(narrator) {Nipote} means both nephew or niece, and grandchild.
(prompt) Say: my grandmother has eight grandchildren. || Mia nonna ha otto nipoti.
(prompt) Say: my cousins live in Sicily. || I miei cugini abitano in Sicilia.
(prompt) Ask a friend: can you give me your number? || Mi dai il tuo numero?
(prompt) Say: we have to stay in touch. || Dobbiamo restare in contatto.
(prompt) Ask: do you still get along with your sister? || Vai ancora d'accordo con tua sorella?

(section) Part 2 — Love and partners
(narrator) Now love. A boyfriend is {il fidanzato}. A girlfriend is {la fidanzata}. To be together is {stare insieme}.
(prompt) Ask a woman friend: are you still with Paolo? Use {fidanzata}. || Sei ancora fidanzata con Paolo?
(narrator) To break up is {lasciarsi}. In the past, it is {ci siamo lasciati}.
(build) sciati | lasciati | ci siamo lasciati
(prompt) Say: we broke up two years ago. || Ci siamo lasciati due anni fa.
(prompt) Say: now I am with Andrea. || Adesso sto con Andrea.
(prompt) Say: we are getting married in September! || Ci sposiamo a settembre!
(prompt) Say: we met at university. Use {conoscersi}. || Ci siamo conosciuti all'università.
(prompt) Say: it was love at first sight. || È stato un colpo di fulmine.
(narrator) After {sono contento che}, use the subjunctive.
(repeat x2 slow) Sono proprio contento che tu sia felice.
(prompt) Say to a friend: I am really happy that you are happy. || Sono proprio contento che tu sia felice.
(prompt) Say: I am happy that you came. Speak informally. || Sono contento che tu sia venuto.
(narrator) To miss someone, Italians say it the other way round. I miss you: {mi manchi}.
(repeat x2) Mi manchi.
(prompt) Say to your partner: I miss you so much. || Mi manchi tanto.
(prompt) Say: we had an argument, but then we made up. Use {litigare} and {fare pace}. || Abbiamo litigato, ma poi abbiamo fatto pace.
(prompt) Say: we broke up two years ago. || Ci siamo lasciati due anni fa.
(prompt) Say: we love each other very much. || Ci vogliamo molto bene.

(section) Part 3 — Classmates and dorm-mates
(narrator) Now school. A classmate at university is {un compagno di corso}. A roommate in a dorm is {un compagno di stanza}.
(prompt) Ask a classmate: can you lend me your notes? || Mi presti i tuoi appunti?
(prompt) Say: I missed the lesson yesterday. || Ieri ho perso la lezione.
(prompt) Ask: shall we study together for the exam? || Studiamo insieme per l'esame?
(prompt) Say: my roommate is very messy. Use {disordinato}. || Il mio compagno di stanza è molto disordinato.
(narrator) To ask nicely, use the conditional: {potresti}. Could you?
(prompt) Ask your roommate: could you turn down the music? || Potresti abbassare la musica?
(prompt) Say: I have an exam tomorrow morning. || Domani mattina ho un esame.
(prompt) Say: let's take turns cleaning. Use {fare i turni}. || Facciamo i turni per pulire.
(prompt) Say: the neighbours complained about the noise. Use {i vicini di casa}. || I vicini di casa si sono lamentati del rumore.
(prompt) Say to a friend: I am happy that you came. || Sono contento che tu sia venuto.
(prompt) Say: we met at university. || Ci siamo conosciuti all'università.

(section) Part 4 — Boss and colleagues
(narrator) Now work. The boss is {il capo}, or more formally {il datore di lavoro}. A colleague is {un collega}.
(prompt) Say: my boss is very demanding. || Il mio capo è molto esigente.
(prompt) Say: but my colleagues are nice. || Ma i miei colleghi sono simpatici.
(narrator) With your boss, you usually use {Lei}. With colleagues, often {tu}.
(prompt) Ask your boss, formally: could I take a day off on Friday? Use {un giorno di ferie}. || Potrei prendere un giorno di ferie venerdì?
(prompt) Ask your boss, formally: can I work from home tomorrow? || Posso lavorare da casa domani?
(prompt) Say: I would like to talk to you about my salary. Formal. || Vorrei parlarle del mio stipendio.
(prompt) Say: I think I deserve a raise. Use {meritare} and {un aumento}. || Penso di meritare un aumento.
(prompt) The boss says, to you: you did an excellent job. Formal. || Ha fatto un ottimo lavoro.
(prompt) Ask a colleague: shall we have a coffee? || Prendiamo un caffè?
(prompt) Say: my colleague always arrives late to meetings. || Il mio collega arriva sempre in ritardo alle riunioni.
(prompt) Ask your roommate: could you turn down the music? || Potresti abbassare la musica?
(prompt) Say: I miss you so much. || Mi manchi tanto.

(section) Part 5 — Landlord and tenant
(narrator) The landlord is {il padrone di casa}. The tenant is {l'inquilino}.
(build) lino | quilino | inquilino
(prompt) Say to your landlord, formally: the tap in the kitchen is leaking. || Il rubinetto in cucina perde.
(prompt) Ask, formally: could you send a plumber? || Potrebbe mandare un idraulico?
(prompt) Say: the heating hasn't worked for three days. || Il riscaldamento non funziona da tre giorni.
(prompt) The landlord says: the rent is due on the fifth of the month. || L'affitto si paga entro il cinque del mese.
(narrator) To end a lease, the tenant gives notice. {La disdetta}. The notice period is {il preavviso}.
(prompt) Say: I would like to give notice. I am leaving in June. || Vorrei dare la disdetta. Me ne vado a giugno.
(prompt) Ask: how many months' notice do I have to give? || Quanti mesi di preavviso devo dare?
(prompt) Ask, formally: when will you give me back the deposit? || Quando mi restituisce la caparra?
(prompt) Say: I think I deserve a raise. || Penso di meritare un aumento.
(prompt) Ask your boss: could I take a day off on Friday? || Potrei prendere un giorno di ferie venerdì?

(section) Part 6 — Strangers on the move
(narrator) Now travellers. On a train, a bus, a plane, or a ferry, we talk to strangers. Use {Lei}.
(prompt) Say, formally: excuse me, I think this is my seat. || Mi scusi, credo che questo sia il mio posto.
(prompt) Ask, formally: may I see your ticket? || Posso vedere il suo biglietto?
(narrator) To ask permission politely: {le dispiace se}? Do you mind if?
(repeat x2 slow) Le dispiace se apro il finestrino?
(prompt) Ask, formally: do you mind if I open the window? || Le dispiace se apro il finestrino?
(prompt) Ask, formally: could you help me lift the suitcase? || Potrebbe aiutarmi ad alzare la valigia?
(prompt) Ask, formally: do you mind if I sit by the window? || Le dispiace se mi siedo vicino al finestrino?
(prompt) On the bus, ask: excuse me, are you getting off at the next stop? || Scusi, scende alla prossima fermata?
(prompt) On the plane, say: excuse me, I need to get through. || Scusi, devo passare.
(narrator) A ferry is {il traghetto}. When the sea is rough, it is {mosso}. Seasickness is {il mal di mare}.
(prompt) Say: the sea is rough today. || Oggi il mare è mosso.
(prompt) Say: I get seasick. Use {soffrire di}. || Soffro il mal di mare.
(prompt) Say: the heating hasn't worked for three days. || Il riscaldamento non funziona da tre giorni.
(prompt) Ask, formally: do you mind if I open the window? || Le dispiace se apro il finestrino?

(section) Part 7 — People on holiday
(narrator) On holiday, people start conversations easily.
(prompt) Ask a group, informally: where are you from? || Di dove siete?
(prompt) Ask: are you here on holiday? || Siete qui in vacanza?
(prompt) Ask: how long have you been here? || Da quanto tempo siete qui?
(prompt) Answer: we've been here for a week. || Siamo qui da una settimana.
(narrator) To switch from {Lei} to {tu}, Italians ask: {ci diamo del tu?}
(prompt) Ask: shall we use "tu" with each other? || Ci diamo del tu?
(prompt) Suggest: why don't we have dinner together tonight? || Perché non ceniamo insieme stasera?
(prompt) Say: it was a pleasure to meet you all. || È stato un piacere conoscervi.
(prompt) Say: we have to stay in touch. || Dobbiamo restare in contatto.
(prompt) Say, formally: excuse me, I think this is my seat. || Mi scusi, credo che questo sia il mio posto.

(section) Part 8 — A strike
(narrator) Strikes are common in Italy. A strike is {uno sciopero}. A protest march is {una manifestazione}. The unions are {i sindacati}.
(build) cati | sindacati
(prompt) Say: today the bus drivers are on strike. || Oggi gli autisti degli autobus sono in sciopero.
(prompt) Say: the unions called a strike for Friday. Use {indire}. || I sindacati hanno indetto uno sciopero per venerdì.
(prompt) Ask a striker, formally: why are you on strike? || Perché siete in sciopero?
(prompt) Answer: we are asking for higher salaries. || Chiediamo stipendi più alti.
(prompt) Answer: and better working conditions. || E condizioni di lavoro migliori.
(prompt) Say: I agree with them. || Sono d'accordo con loro.
(prompt) Say: but I can't get to work today. || Ma oggi non posso arrivare al lavoro.
(prompt) Ask: how long have you been here? || Da quanto tempo siete qui?
(prompt) Say: I would like to give notice. || Vorrei dare la disdetta.

(section) Part 9 — At the football match
(narrator) Now football, {il calcio}. A fan is {un tifoso}. To support a team is {tifare per}.
(prompt) Ask a friend: which team do you support? || Per che squadra tifi?
(prompt) Answer: I support Roma. || Tifo per la Roma.
(prompt) Say: what a match! || Che partita!
(prompt) Say: the referee didn't see the foul. Use {l'arbitro} and {il fallo}. || L'arbitro non ha visto il fallo.
(prompt) Shout: it was a penalty! || Era rigore!
(prompt) Say: we won three to one. || Abbiamo vinto tre a uno.
(prompt) Say: last week we drew. Use {pareggiare}. || La settimana scorsa abbiamo pareggiato.
(prompt) Say: the stadium was full. || Lo stadio era pieno.
(prompt) Say: today the bus drivers are on strike. || Oggi gli autisti degli autobus sono in sciopero.

(section) Part 10 — A children's birthday party
(narrator) Finally, a birthday party for your child's friend. A little friend, for children, is {un amichetto}, or {un'amichetta}.
(prompt) Ask a woman, informally: are you Sofia's mum? || Sei la mamma di Sofia?
(prompt) Say: thank you for the invitation. || Grazie per l'invito.
(prompt) Say: my son is allergic to peanuts. Use {le arachidi}. || Mio figlio è allergico alle arachidi.
(prompt) Ask: what time should I come and pick him up? || A che ora devo venire a prenderlo?
(prompt) Say to the birthday girl: happy birthday! || Buon compleanno!
(prompt) Say: now blow out the candles! Speak to a child. || Adesso spegni le candeline!
(prompt) Say: what a lovely party! || Che bella festa!
(prompt) Say: the children had a great time. Use {divertirsi}. || I bambini si sono divertiti tantissimo.
(prompt) Say: next time, the party is at our house. || La prossima volta la festa è a casa nostra.
(prompt) Ask a friend: which team do you support? || Per che squadra tifi?

(section) Part 11 — Review
(narrator) Let's review. First, sentences from earlier lessons.
(prompt) Say: I would like to do my thesis on photonics. || Vorrei fare la tesi sulla fotonica.
(prompt) Say: it is better to buy online. || Conviene comprare online.
(prompt) Say: I waited in line for two hours. || Ho fatto la fila per due ore.
(prompt) Say: I thought it was still yellow. || Pensavo che fosse ancora giallo.
(prompt) Say: I would like full insurance, without excess. || Vorrei l'assicurazione completa, senza franchigia.
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: we haven't seen each other for ten years! || Non ci vediamo da dieci anni!
(prompt) Say: we love each other very much. || Ci vogliamo molto bene.
(prompt) Say: we broke up two years ago. || Ci siamo lasciati due anni fa.
(prompt) Say: I am really happy that you are happy. || Sono proprio contento che tu sia felice.
(prompt) Ask a classmate: can you lend me your notes? || Mi presti i tuoi appunti?
(prompt) Ask your boss: could I take a day off on Friday? || Potrei prendere un giorno di ferie venerdì?
(prompt) Say to your landlord: the tap in the kitchen is leaking. || Il rubinetto in cucina perde.
(prompt) Ask, formally: do you mind if I open the window? || Le dispiace se apro il finestrino?
(prompt) Ask: shall we use "tu" with each other? || Ci diamo del tu?
(prompt) Say: the unions called a strike for Friday. || I sindacati hanno indetto uno sciopero per venerdì.
(prompt) Say: the referee didn't see the foul. || L'arbitro non ha visto il fallo.
(prompt) Say: the children had a great time. || I bambini si sono divertiti tantissimo.

(narrator) Now you are Luca. Answer Chiara.
(it F) Luca! Che sorpresa! Questo posto è libero?
(prompt M) Say: of course, sit down. We haven't seen each other for ten years! || Certo, siediti. Non ci vediamo da dieci anni!
(it F) Lo so! Io adesso lavoro in una banca a Milano.
(prompt M) Ask: are you still with Paolo? || Sei ancora fidanzata con Paolo?
(it F) No, ci siamo lasciati. Adesso sto con Andrea, e ci sposiamo a settembre!
(prompt M) Say: come on! Congratulations! I'm really happy that you're happy. || Ma dai! Congratulazioni! Sono proprio contento che tu sia felice.
(it F) Grazie! Dobbiamo restare in contatto.
(prompt M) Say: of course. I'll write to you on WhatsApp tonight. || Certo. Ti scrivo su WhatsApp stasera.

(narrator) Now listen to the first conversation again.
(pause 1)
(it M) Chiara? Sei proprio tu? Non ci vediamo da dieci anni!
(it F) Luca! Che sorpresa! Questo posto è libero?
(it M) Certo, siediti. Allora, come stai? Che cosa fai adesso?
(it F) Lavoro in una banca a Milano. Il mio capo è molto esigente, ma i colleghi sono simpatici.
(it M) E sei ancora fidanzata con Paolo?
(it F) No, ci siamo lasciati due anni fa. Adesso sto con Andrea, e ci sposiamo a settembre!
(it M) Ma dai! Congratulazioni! Sono proprio contento che tu sia felice.
(it F) Grazie! E tu? Sei sposato?
(it M) Sì, e abbiamo due figli. Oggi vado alla festa di compleanno di un amichetto di mio figlio.
(it F) Che bello! E vai ancora d'accordo con tua sorella?
(it M) Sì, ci vogliamo molto bene. Con mia suocera, invece, è un'altra storia!
(it F) Dai, dobbiamo restare in contatto. Mi dai il tuo numero?
(it M) Certo. Ti scrivo su WhatsApp stasera.
(pause 1.5)
(narrator) This is the end of lesson five. Listen to it again tomorrow. {Un abbraccio, e a presto!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
